# nb01 - Spark 4.x tour: what changed since 3.5

This stack runs **Apache Spark 4.2.0** (released 2026-07-14) - the third release of the 4.x
line - with **Delta Lake 4.4.1** on Java 21 / Scala 2.13 / Python 3.

If your muscle memory comes from Spark 3.5, the single biggest behavioral change is:
**ANSI mode is on by default**. `CAST` now *throws* instead of silently returning NULL.
This notebook tours the new defaults and the newest SQL surface before we get our hands
dirty in nb02.

**Useful URLs while you work:**

| Where | URL |
|-------|-----|
| Master UI (cluster) | http://localhost:8080 |
| Worker UI | http://localhost:8081 |
| Driver UI (this notebook, while a job runs) | http://localhost:4041 |
| History Server (finished jobs - keep open in a 2nd tab) | http://localhost:18080 |
| Spark Connect (`sc://spark-connect:15002`) | used in the last section |

Prerequisite: the stack is up (`docker compose ps` shows `spark-master` healthy).
No seed data needed for this notebook.


## 1. Session + defaults audit

`de_stack.get_spark()` builds our standard session: it points at the cluster master
(`spark://spark-master:7077`) and pre-enables the Delta extensions.


In [1]:
from de_stack import get_spark, CONNECT_URL, sql

spark = get_spark("nb01-tour")
print("Spark version :", spark.version)
print("Master        :", spark.sparkContext.master)

defaults = [
    "spark.sql.ansi.enabled",              # <-- 4.0: ON by default (was OFF)
    "spark.sql.session.timeZone",
    "spark.sql.catalogImplementation",
    "spark.sql.adaptive.enabled",          # AQE: on by default since 3.2
    "spark.sql.execution.arrow.pyspark.enabled",  # <-- 4.2: Arrow Python UDFs ON by default
    "spark.sql.sources.default",           # default file format for CREATE TABLE
    "spark.sql.warehouse.dir",
]
for k in defaults:
    try:
        print(f"{k:52s} = {spark.conf.get(k)}")
    except Exception:
        print(f"{k:52s} = (unset)")


11:48:42.164 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version : 4.2.0
Master        : spark://spark-master:7077
spark.sql.ansi.enabled                               = true
spark.sql.session.timeZone                           = Etc/UTC
spark.sql.catalogImplementation                      = in-memory
spark.sql.adaptive.enabled                           = true
spark.sql.execution.arrow.pyspark.enabled            = true
spark.sql.sources.default                            = parquet
spark.sql.warehouse.dir                              = file:/data/warehouse


## 2. ANSI mode: `CAST` throws now

In 3.5 this was the classic silent-data-corruption trap: `CAST('abc' AS INT)` -> `NULL`.
Under ANSI (4.0+ default) it raises `SparkNumberFormatException`. The clean migration is
**`try_cast`** (returns NULL) or **`try_*`** variants of functions - not switching ANSI off.


In [2]:
# 1) the ANSI way: it errors, and the error names the offending value
try:
    spark.sql("SELECT CAST('abc' AS INT) AS parsed").collect()
except Exception as e:
    print(type(e).__name__, "->", str(e).splitlines()[0][:130])

# 2) try_cast: the explicit "give me NULL on garbage" contract
spark.sql("SELECT try_cast('abc' AS INT) AS parsed, try_cast('42' AS INT) AS ok").show()

# 3) try_divide: division by zero -> NULL instead of an exception
spark.sql("SELECT try_divide(10, 3) AS fine, try_divide(10, 0) AS safe_zero").show()


{"ts": "2026-10-13 11:48:47.660", "level": "ERROR", "logger": "SQLQueryContextLogger", "msg": "[CAST_INVALID_INPUT] The value 'abc' of the type \"STRING\" cannot be cast to \"INT\" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018", "context": {"errorClass": "CAST_INVALID_INPUT"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o48.collectToPython.\n: org.apache.spark.SparkNumberFormatException: [CAST_INVALID_INPUT] The value 'abc' of the type \"STRING\" cannot be cast to \"INT\" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018\n== SQL (line 1, position 8) ==\nSELECT CAST('abc' AS INT) AS parsed\n       ^^^^^^^^^^^^^^^^^^\n\n\tat org.apache.spark.sql.errors.QueryExecutionErrors$.invalidInputInCastToNumberError(Q

NumberFormatException -> [CAST_INVALID_INPUT] The value 'abc' of the type "STRING" cannot be cast to "INT" because it is malformed. Correct the value as pe


+------+---+
|parsed| ok|
+------+---+
|  NULL| 42|
+------+---+

+------------------+---------+
|              fine|safe_zero|
+------------------+---------+
|3.3333333333333335|     NULL|
+------------------+---------+



> **Study note:** every notebook in this course runs with ANSI on, and our bronze data
> (nb02) contains dirty values on purpose - `quantity` sometimes arrives as a *string*.
> ANSI forces the pipeline to declare how bad values are handled instead of leaking NULLs.


## 3. SQL pipe syntax (new in 4.0)

`|>` lets you read a query top-to-bottom, in the order you think about it:
source -> filter -> project -> group -> sort. No more writing the `SELECT` list before
you know what you are filtering on.


In [3]:
spark.sql('''
    SELECT * FROM VALUES
        ('DE', 'EMEA', 100.0), ('US', 'NA', 250.0), ('FR', 'EMEA', 75.0), ('GB', 'EMEA', 500.0)
    AS t(country, region, amount)
''').createOrReplaceTempView("orders_demo")

spark.sql('''
    FROM orders_demo
    |> WHERE amount > 50
    |> SELECT region, SUM(amount) AS total, COUNT(*) AS orders GROUP BY region
    |> ORDER BY total DESC
''').show()


+------+-----+------+
|region|total|orders|
+------+-----+------+
|  EMEA|675.0|     3|
|    NA|250.0|     1|
+------+-----+------+



## 4. `QUALIFY` (new in 4.2)

Filtering window-function results normally needs a subquery or CTE. `QUALIFY` applies the
condition to the window/aggregation result directly - one less nesting level for the
"top-N per group" pattern you will use constantly in silver/gold layers.


In [4]:
# best order per region, without a subquery
spark.sql('''
    SELECT country, region, amount
    FROM orders_demo
    QUALIFY ROW_NUMBER() OVER (PARTITION BY region ORDER BY amount DESC) = 1
''').show()


+-------+------+------+
|country|region|amount|
+-------+------+------+
|     GB|  EMEA| 500.0|
|     US|    NA| 250.0|
+-------+------+------+



## 5. Run a job and look at the UI

Spark 4.2 ships a **modernized Web UI** (dark mode, searchable & zoomable plan
visualization, side-by-side initial-vs-final plan comparison for AQE queries, job timeline).
Kick a real job, then open the links - the History Server keeps every finished job, so
you can study plans after the fact.


In [5]:
spark.sql("SELECT count(*) AS n FROM range(20000000)").show()

print("driver UI (inside network):", spark.sparkContext.uiWebUrl)
print("driver UI (from your host): http://localhost:4041")
print("history server             : http://localhost:18080")


+--------+
|       n|
+--------+
|20000000|
+--------+

driver UI (inside network): http://jupyter:4040
driver UI (from your host): http://localhost:4041
history server             : http://localhost:18080


## 6. Spark Connect: the same API, thin client

`spark-submit`/classic drivers ship the whole engine to your JVM. **Spark Connect** (mature
since 4.0, and where new development happens) keeps only a gRPC client in your process:
the session above runs *your* SQL locally; this next one ships plans to the
`spark-connect` service at `sc://spark-connect:15002`. Same Python API, different
execution location - this is how notebooks/services in production should talk to clusters.


In [6]:
remote = get_spark("nb01-connect", remote=CONNECT_URL)

print("remote Spark:", remote.version)
remote.sql("SELECT 'thin client' AS mode, 6 * 7 AS answer").show()

# DataFrames over Connect are just handles to a plan stored on the server
df = remote.sql("SELECT id FROM range(100000)")
print("DataFrame type:", type(df).__module__)
df.groupBy().sum("id").show()


remote Spark: 4.2.0
+-----------+------+
|       mode|answer|
+-----------+------+
|thin client|    42|
+-----------+------+

DataFrame type: pyspark.sql.classic.dataframe


11:48:52.972 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


+----------+
|   sum(id)|
+----------+
|4999950000|
+----------+



## Wrap-up

You now know the environment and the three 4.x changes you will trip over first:
ANSI-by-default, pipe syntax, and `QUALIFY`. Next: **nb02_bronze_to_silver.ipynb** -
cleaning deliberately messy real-shaped data into a silver Delta layer (it auto-seeds
the bronze files if you have not run `make seed`).
